# Chronos Scramble v3 — IQSP Gaussian Protocol (Safe TPU Calibration)
### 4-Stream Covariance Matrix Capture with λ-Interpolation Schedule

**Improvements over v2:**
- **True TPU Calibration**: Calibration now directly uses `jax.pmap`, guaranteeing that matrices are properly sized for the TPU and don't cause 112-second rolling window smearing.
- **Diff Timestamp Recovery**: `io_callback` timestamps are natively differentiated to extract exact hardware execution jitter rather than relying on absolute timestamp covariance.
- **Dry Run Safety**: A short duration test block allows you to verify it is working perfectly before committing the full 30-minute block.

> **Step 1: Set role. Step 2: Run Calibration to hit 50 Hz. Step 3: Run the Short Dry Run. Step 4: Run the 30-min Capture.**


In [ ]:
#@title 1 · Experiment Configuration { display-mode: "form" }
EXPERIMENT_ROLE   = 'Alice_Scramble' #@param ["Alice_Scramble", "Bob_Passive"]
DURATION_SECONDS  = 1800.0           #@param {type:"number"}  # 30 min = 3 full cycles
N_STREAMS         = 4                #@param {type:"number"}
CHUNK_SECONDS     = 5                #@param {type:"number"}  # NPZ chunk duration
WINDOW_PACKETS    = 32               #@param {type:"number"}  # rolling cov window
SIGMA_GAUSSIAN    = 1.0              #@param {type:"number"}  # Gaussian width across streams
BASE_MATRIX_SIZE  = 256              #@param {type:"number"}  # overridden by calibration
TARGET_LOOP_HZ    = 50.0             #@param {type:"number"}  # target per-stream loop rate

# λ schedule: 3min ramp-up, 3min ON, 1min ramp-down, 3min OFF  (10 min/cycle × 3 cycles)
RAMP_UP_S   = 180.0  #@param {type:"number"}
ON_S        = 180.0  #@param {type:"number"}
RAMP_DOWN_S =  60.0  #@param {type:"number"}
OFF_S       = 180.0  #@param {type:"number"}

import os, time, json, pathlib, shutil, threading
import numpy as np

CYCLE_S = RAMP_UP_S + ON_S + RAMP_DOWN_S + OFF_S
OUTPUT_DIR = f'/tmp/chronos_v3_{EXPERIMENT_ROLE.lower()}'
pathlib.Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print(f'Role:          {EXPERIMENT_ROLE}')
print(f'Duration:      {DURATION_SECONDS:.0f}s  ({DURATION_SECONDS/60:.1f} min)')
print(f'Cycle:         {CYCLE_S:.0f}s  ({CYCLE_S/60:.1f} min)  ×  {DURATION_SECONDS/CYCLE_S:.1f} cycles')


In [ ]:
#@title 2 · Infrastructure Fingerprint (datacenter / zone detection)
import socket, platform, subprocess
import urllib.request, json as _json

infra = {}
GCP_META    = 'http://metadata.google.internal/computeMetadata/v1'
GCP_HEADERS = {'Metadata-Flavor': 'Google'}

def gcp_meta(path):
    try:
        req = urllib.request.Request(f'{GCP_META}/{path}', headers=GCP_HEADERS)
        return urllib.request.urlopen(req, timeout=3).read().decode().strip()
    except: return None

zone_full = gcp_meta('instance/zone')
infra['gcp_zone']         = zone_full.split('/')[-1] if zone_full else None
infra['gcp_machine_type'] = (gcp_meta('instance/machine-type') or '').split('/')[-1] or None
infra['gcp_instance_id']  = gcp_meta('instance/id')

try:
    geo = _json.loads(urllib.request.urlopen('https://ipinfo.io/json', timeout=5).read())
    infra.update({k: geo.get(k) for k in ['ip','city','region','country','timezone','loc','org']})
except Exception as e: infra['geo_error'] = str(e)

infra['hostname']  = socket.gethostname()
infra['cpu_count'] = os.cpu_count()

print('='*55)
print('INFRASTRUCTURE FINGERPRINT')
print('='*55)
for k,v in infra.items(): print(f'  {k:<22}: {v}')
print('='*55)
print()
print('>>> Compare GCP zone and ip with the OTHER session.')
print('>>> They MUST be different zones before proceeding.')


In [ ]:
#@title 3 · TPU pmap Engine + True Calibration (run BEFORE the other session)
# Defines the true JAX hardware kernel and tests IT, not numpy.
try:
    import jax, jax.numpy as jnp
    from jax.experimental import io_callback
    n_devices = len(jax.devices())
    USE_JAX = True
    print(f'JAX backend: {jax.default_backend()}  devices: {n_devices}')
except Exception as e:
    USE_JAX = False
    print(f'JAX unavailable ({e}), using CPU numpy fallback.')

# ── Device-side timed kernel ───────────────────────────────
if USE_JAX:
    _ts_lock   = threading.Lock()
    _ts_store  = {k: [] for k in range(N_STREAMS)}
    def _record_ts(stream_k, t_arr):
        with _ts_lock:
            _ts_store[int(stream_k)].append(float(t_arr))
    def make_jax_kernel(stream_k, mat_size):
        @jax.jit
        def _kernel(key):
            A = jax.random.normal(key, (mat_size, mat_size), dtype=jnp.float32)
            B = jax.random.normal(jax.random.fold_in(key, 1), (mat_size, mat_size), dtype=jnp.float32)
            C = A @ B
            t_now = io_callback(lambda: np.float32(time.perf_counter()), jax.ShapeDtypeStruct((), jnp.float32))
            io_callback(lambda t: _record_ts(stream_k, t), None, t_now)
            return C.mean()
        return _kernel
    
    # JIT compiler cache
    _kern_cache = {}
    def get_kern(k, size):
        if (k, size) not in _kern_cache:
            kern = make_jax_kernel(k, size)
            kern(jax.random.PRNGKey(0)) # warmup
            _kern_cache[(k, size)] = kern
        return _kern_cache[(k, size)]

def execute_packet(sizes, packet_n):
    """Runs one full step across all 4 streams, returning execution durations in ms."""
    dt_ms = []
    if USE_JAX:
        futures = []
        for k in range(N_STREAMS):
            kern = get_kern(k, sizes[k])
            key  = jax.random.PRNGKey(packet_n * N_STREAMS + k)
            futures.append((k, kern(key)))
        jax.effects_barrier()
        for k, _ in futures:
            with _ts_lock:
                ts_list = _ts_store[k]
                t_abs = ts_list.pop(0) if ts_list else 0.0
            # We keep the absolute timestamps to differentiate later
            dt_ms.append(t_abs)
    else:
        for size in sizes:
            t1 = time.perf_counter()
            np.random.randn(size, size).astype(np.float32) @ np.random.randn(size, size).astype(np.float32)
            dt_ms.append((time.perf_counter() - t1))
    return dt_ms

# ── True Calibration Loop ───────────────────────────────────────────────
print('\nStarting True Hardware Calibration... (may take 2 minutes for JIT)')
target_ms = 1000.0 / TARGET_LOOP_HZ
lo, hi = 32, 1024
for step in range(8):
    mid = (lo + hi) // 2
    # Run 10 packets
    times = []
    for pn in range(10):
        t_abs = execute_packet([mid]*N_STREAMS, pn)
        times.append(t_abs)
    # Differentiate absolute timestamps to get durations
    dts = np.diff(np.array(times), axis=0) * 1000.0
    dt = np.mean(dts) if len(dts) > 0 else 0
    print(f'  Size {mid:4d}x{mid:<4d} -> {dt:5.1f}ms')
    if dt < target_ms: lo = mid
    else: hi = mid

BASE_MATRIX_SIZE = lo
print(f'\n>> CALIBRATED BASE SIZE: {BASE_MATRIX_SIZE}')

# Measure exact baseline stats
cal_times = []
for pn in range(50):
    cal_times.append(execute_packet([BASE_MATRIX_SIZE]*N_STREAMS, pn+100))

dts = np.diff(np.array(cal_times), axis=0) * 1000.0
BASELINE_MEAN = np.mean(dts, axis=0)
BASELINE_STD  = np.std(dts, axis=0)
BASELINE_COV  = np.cov(dts, rowvar=False)

actual_hz = 1000.0 / BASELINE_MEAN.mean()
print(f'Baseline mean loop rate: {actual_hz:.1f} Hz  (target={TARGET_LOOP_HZ:.0f} Hz)')
go = abs(actual_hz - TARGET_LOOP_HZ) / TARGET_LOOP_HZ < 0.5
print('>>> ' + ('GO ✓ — rates within 50% of target.' if go else 'WAIT ✗ — adjust TARGET_LOOP_HZ and rerun.'))


In [ ]:
#@title 4 · λ-Schedule and Gaussian Profile
def compute_lambda(elapsed):
    phase = elapsed % CYCLE_S
    if phase < RAMP_UP_S:
        return 0.5 * (1 - np.cos(np.pi * phase / RAMP_UP_S))
    elif phase < RAMP_UP_S + ON_S:
        return 1.0
    elif phase < RAMP_UP_S + ON_S + RAMP_DOWN_S:
        p = (phase - RAMP_UP_S - ON_S) / RAMP_DOWN_S
        return 0.5 * (1 + np.cos(np.pi * p))
    else: return 0.0

STREAM_MU = (N_STREAMS - 1) / 2.0
STREAM_PROFILE = np.array([np.exp(-0.5 * ((k - STREAM_MU) / SIGMA_GAUSSIAN)**2) for k in range(N_STREAMS)], dtype=np.float32)
STREAM_PROFILE /= STREAM_PROFILE.max()

def gaussian_amplitudes(lam):
    return lam * STREAM_PROFILE
print('Schedule loaded.')


In [ ]:
#@title 5 · The Capture Engine (Do not run this directly)
SIZE_RANGE = max(32, BASE_MATRIX_SIZE // 2)
import shutil

def run_capture_loop(duration, prefix_dir):
    out_dir = f'/tmp/{prefix_dir}'
    pathlib.Path(out_dir).mkdir(parents=True, exist_ok=True)
    
    chunks_written = 0
    chunk_manifest = []
    stream_abs_buffers = [[] for _ in range(N_STREAMS)]
    stream_sizes_buf   = [[] for _ in range(N_STREAMS)]
    packet_n = 0
    t0 = time.time()
    chunk_start = t0
    
    print(f'Starting {duration}s capture...')
    while time.time() - t0 < duration:
        elapsed = time.time() - t0
        lam = compute_lambda(elapsed)
        amps = gaussian_amplitudes(lam)
        
        sizes = [(BASE_MATRIX_SIZE + int(amps[k] * SIZE_RANGE) if EXPERIMENT_ROLE == 'Alice_Scramble' else BASE_MATRIX_SIZE) for k in range(N_STREAMS)]
        
        # Execute packet
        t_abs_list = execute_packet(sizes, packet_n)
        for k in range(N_STREAMS):
            stream_abs_buffers[k].append(t_abs_list[k])
            stream_sizes_buf[k].append(sizes[k])
            
        packet_n += 1
        
        if time.time() - chunk_start >= CHUNK_SECONDS:
            chunk_mid = (chunk_start + time.time()) / 2.0
            min_len = min(len(b) for b in stream_abs_buffers)
            if min_len >= WINDOW_PACKETS + 1:
                # Differentiate the rolling absolute timestamps immediately
                abs_arr = np.array([b[-(WINDOW_PACKETS+1):] for b in stream_abs_buffers], dtype=np.float64)
                timing_dt = np.diff(abs_arr, axis=1) * 1000.0
                cov_mat   = np.cov(timing_dt).astype(np.float32)
                sizes_arr = np.array([s[-WINDOW_PACKETS:] for s in stream_sizes_buf], dtype=np.int32)
                lam_mid   = float(compute_lambda(chunk_mid - t0))
                
                fname = f'chunk_{chunks_written:05d}.npz'
                np.savez_compressed(
                    pathlib.Path(out_dir) / fname,
                    host_time_mid     = np.float64(chunk_mid),
                    stream_timing     = timing_dt.astype(np.float32),
                    covariance_matrix = cov_mat,
                    lambda_val        = np.float32(lam_mid),
                    stream_sizes      = sizes_arr,
                    baseline_cov      = BASELINE_COV.astype(np.float32),
                    use_jax           = np.bool_(USE_JAX),
                )
                chunk_manifest.append({'chunk': fname, 'host_time_mid': float(chunk_mid), 'lambda_val': lam_mid, 'n_packets': WINDOW_PACKETS})
                chunks_written += 1
                
                # Keep overlap for contiguous diffs
                for k in range(N_STREAMS):
                    stream_abs_buffers[k] = stream_abs_buffers[k][-(WINDOW_PACKETS+1):]
                    stream_sizes_buf[k]   = stream_sizes_buf[k][-WINDOW_PACKETS:]
                chunk_start = time.time()
                if chunks_written % 12 == 0:
                    print(f'  [{elapsed:6.0f}s] chunk={chunks_written:3d}  λ={lam_mid:.3f}')
                    
    print(f'Finished {chunks_written} chunks.')
    return chunk_manifest, packet_n, out_dir, t0


In [ ]:
#@title 6 · DRY RUN (1 Minute Test) - RUN THIS BEFORE THE 30 MIN BLOCK
# This verifies your TPUs won't smear before you commit to 30 min.
print('Starting 60s dry run...')
mani, pkts, d_dir, d_t0 = run_capture_loop(60, f'chronos_v3_dryrun_{EXPERIMENT_ROLE.lower()}')
print(f'Dry run generated {len(mani)} chunks. Average loop rate: {pkts/60.0:.1f} Hz')
if pkts/60.0 < 10:
    print('WARNING: LOOP RATE IS BELOW 10 HZ. Check calibration!')
else:
    print('SUCCESS: System is healthy. Ready for Cell 7.')


In [ ]:
#@title 7 · Main Capture (30 min) + Download
mani, pkts, final_dir, final_t0 = run_capture_loop(DURATION_SECONDS, f'chronos_v3_{EXPERIMENT_ROLE.lower()}')

manifest = {
    'schema_version': '3.0',
    'dataset_type': 'gaussian_covariance_stream_v3',
    'role': EXPERIMENT_ROLE,
    't0_utc': final_t0,
    'duration_s': DURATION_SECONDS,
    'n_chunks': len(mani),
    'n_streams': N_STREAMS,
    'chunk_seconds': CHUNK_SECONDS,
    'sigma_gaussian': SIGMA_GAUSSIAN,
    'stream_profile': STREAM_PROFILE.tolist(),
    'cycle_s': CYCLE_S,
    'ramp_up_s': RAMP_UP_S,
    'on_s': ON_S,
    'ramp_down_s': RAMP_DOWN_S,
    'off_s': OFF_S,
    'base_matrix_size': BASE_MATRIX_SIZE,
    'size_range': SIZE_RANGE if EXPERIMENT_ROLE == 'Alice_Scramble' else 0,
    'infrastructure': infra,
    'baseline_mean': BASELINE_MEAN.tolist() if isinstance(BASELINE_MEAN, np.ndarray) else [],
    'baseline_std': BASELINE_STD.tolist() if isinstance(BASELINE_STD, np.ndarray) else [],
    'chunk_files': [c['chunk'] for c in mani],
    'chunks': mani,
}
with open(f'{final_dir}/manifest.json', 'w') as f: json.dump(manifest, f, indent=2)

archive_name = f'chronos_v3_{EXPERIMENT_ROLE.lower()}'
shutil.make_archive(f'/tmp/{archive_name}', 'zip', final_dir)
print(f'Archive: /tmp/{archive_name}.zip')
try:
    from google.colab import files
    files.download(f'/tmp/{archive_name}.zip')
except: pass
